
# 🇹🇭 Thai TTS Fine-tuning (MMS-TTS → VITS) — Google Colab Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samiabat/thai-colab/blob/main/Thai_TTS_Finetune_MMS_Colab.ipynb)

This notebook fine-tunes **Meta MMS-TTS (Thai)** on your own dataset using the **`finetune-hf-vits`** recipe.
It assumes you already have:
- a folder of WAV files (ideally mono 16 kHz), and
- a transcript file (CSV/TSV) or a way to map each audio to its Thai text.

> **Note:** MMS-TTS is released under **CC-BY-NC 4.0**. If you need commercial use, consider training from scratch or a different base.

## 🚀 Quick Setup Guide
1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → GPU (T4/V100/A100)
2. **Mount Drive**: Run the Google Drive mount cell below
3. **Configure Paths**: Update the data paths in the configuration section
4. **Run All Cells**: Execute cells in order

## ✅ Fixed Issues
- ✅ Fixed `monotonic_align` compilation error
- ✅ Fixed column name typo (`fila_name` → `file_name`)
- ✅ Fixed audio column configuration
- ✅ Added proper error handling and validation


In [1]:

import torch, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
else:
    print('⚠️ No GPU detected. In Colab: Runtime → Change runtime type → T4/V100/A100 GPU')


Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Platform: Linux-6.1.123+-x86_64-with-glibc2.35
CUDA available: True
CUDA device: Tesla T4


In [2]:

!pip -q install --upgrade pip
# Core
!pip -q install torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers accelerate datasets soundfile librosa pythainlp pandas jiwer
# Recipe
!git clone -q https://github.com/ylacombe/finetune-hf-vits.git
%cd finetune-hf-vits
!pip -q install -r requirements.txt
# Compile monotonic_align (fix for training error)
!python setup.py build_ext --inplace
%cd -


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.1 MB/s eta 0:00:00
/content/finetune-hf-vits
/content


In [3]:

from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')


Mounted at /content/drive
✅ Google Drive mounted



## Configure your paths

- `DATA_ROOT`: the directory containing your WAV files (recursively scanned).
- Either provide `TRANSCRIPT_CSV` (with `path,text`) **or** let the notebook build one from a folder structure+text file.
- Output processed dataset and configs will be written under `WORK_DIR`.


In [5]:

from pathlib import Path
import os

# ✅ EDIT THESE PATHS FOR YOUR DATASET
DATA_ROOT = Path('/content/drive/MyDrive/cloned-thai-dataset/audio-data')   # folder with your wavs
TRANSCRIPT_CSV = Path('/content/drive/MyDrive/cloned-thai-dataset/metadata.csv')  # CSV with columns: path,text. Set to None if you don't have it.
WORK_DIR = Path('/content/drive/MyDrive/thai_tts/work')    # where to write normalized data, configs, checkpoints

# MMS language code for Thai
LANG_CODE = 'tha'

# Create work directory
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('📁 Configuration:')
print('DATA_ROOT  =', DATA_ROOT)
print('CSV        =', TRANSCRIPT_CSV)
print('WORK_DIR   =', WORK_DIR)

# Basic validation
print('\n🔍 Validation:')
if DATA_ROOT.exists():
    audio_files = list(DATA_ROOT.rglob('*.wav')) + list(DATA_ROOT.rglob('*.mp3'))
    print(f'✅ DATA_ROOT exists with {len(audio_files)} audio files')
else:
    print(f'⚠️ DATA_ROOT does not exist: {DATA_ROOT}')
    print('   Please update the path or upload your dataset to Google Drive')

if TRANSCRIPT_CSV and TRANSCRIPT_CSV.exists():
    print(f'✅ TRANSCRIPT_CSV exists: {TRANSCRIPT_CSV}')
elif TRANSCRIPT_CSV:
    print(f'⚠️ TRANSCRIPT_CSV does not exist: {TRANSCRIPT_CSV}')
    print('   Will attempt to create one from .txt files alongside audio')
else:
    print('ℹ️ No TRANSCRIPT_CSV specified - will create from folder structure')

print(f'✅ WORK_DIR ready: {WORK_DIR}')


DATA_ROOT  = /content/drive/MyDrive/cloned-thai-dataset/audio-data
CSV        = /content/drive/MyDrive/cloned-thai-dataset/metadata.csv
WORK_DIR   = /content/drive/MyDrive/thai_tts/work



### (Optional) Create `metadata.csv` if you don't already have one

If you **don't** have a CSV, this cell creates a simple CSV by scanning for `.wav` files and reading a paired `.txt` with the same basename for text (e.g., `utt001.wav` + `utt001.txt`).  
Adjust as needed for your layout.


In [6]:

import pandas as pd
from pathlib import Path

def build_metadata_from_sidecar_txt(data_root: Path, out_csv: Path):
    rows = []
    for wav in data_root.rglob('*.wav'):
        txt = wav.with_suffix('.txt')
        if txt.exists():
            text = txt.read_text(encoding='utf-8').strip()
            rows.append({'path': str(wav.resolve()), 'text': text})
    if not rows:
        raise ValueError('No (wav, txt) pairs found. Provide TRANSCRIPT_CSV instead.')
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return out_csv

if str(TRANSCRIPT_CSV).lower() == 'none':
    TRANSCRIPT_CSV = WORK_DIR / 'metadata.csv'
    created = build_metadata_from_sidecar_txt(DATA_ROOT, TRANSCRIPT_CSV)
    print('Created CSV at', created)
else:
    print('Using existing metadata CSV:', TRANSCRIPT_CSV)


Using existing metadata CSV: /content/drive/MyDrive/cloned-thai-dataset/metadata.csv



## Preprocess audio & normalize Thai text

- Resample/convert to **mono 16 kHz WAV**
- Light Thai normalization (using PyThaiNLP)
- Filter clips (1–12 s recommended)
- Produce a cleaned `metadata_clean.csv`


In [11]:

import librosa, soundfile as sf, pandas as pd, numpy as np, os
from pythainlp.tokenize import word_tokenize

PROC_AUDIO_DIR = WORK_DIR / 'wavs_16k_mono'
PROC_AUDIO_DIR.mkdir(exist_ok=True, parents=True)
OUT_CSV = WORK_DIR / 'metadata_clean.csv'

MIN_DUR = 1.0
MAX_DUR = 12.0
TARGET_SR = 16000

def normalize_thai(text: str) -> str:
    # Minimal normalization: collapse spaces; optional segmentation (helps prosody)
    seg = word_tokenize(text.strip(), engine='newmm')
    return ' '.join(seg)

df = pd.read_csv(TRANSCRIPT_CSV)
clean_rows = []

for i, row in df.iterrows():
    src = Path('/content/drive/MyDrive/cloned-thai-dataset/' + row['file_name'])
    text = str(row['text'])
    if not src.exists():
        print('Skip (missing):', src)
        continue
    try:
        wav, sr = librosa.load(src, sr=None, mono=True)
        dur = len(wav)/sr
        if dur < MIN_DUR or dur > MAX_DUR:
            continue
        if sr != TARGET_SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=TARGET_SR)
            sr = TARGET_SR
        # Write processed wav under WORK_DIR, mirroring file name
        out_wav = PROC_AUDIO_DIR / f"{src.stem}_16k.wav"
        sf.write(out_wav, wav, sr, subtype='PCM_16')
        clean_rows.append({'file_name': str(out_wav), 'text': normalize_thai(text)})
    except Exception as e:
        print('Error:', src, e)

clean_df = pd.DataFrame(clean_rows)
clean_df.to_csv(OUT_CSV, index=False)
print('Wrote cleaned CSV:', OUT_CSV)
print('Total usable clips:', len(clean_df))
clean_df.head()


Wrote cleaned CSV: /content/drive/MyDrive/thai_tts/work/metadata_clean.csv
Total usable clips: 521


,file_name,text
0,/content/drive/MyDrive/thai_tts/work/wavs_16k_...,เช้า ต้อง เห็น หน้า เย็น ก็ ต้อง ...
1,/content/drive/MyDrive/thai_tts/work/wavs_16k_...,ดิ้น ดิ้น
2,/content/drive/MyDrive/thai_tts/work/wavs_16k_...,กำจัด
3,/content/drive/MyDrive/thai_tts/work/wavs_16k_...,เคาะ ไล่ อากาศ
4,/content/drive/MyDrive/thai_tts/work/wavs_16k_...,เว็บ ดีไซน์



### (Optional) Build a 🤗 Datasets arrow dataset

The `finetune-hf-vits` script can read from a HuggingFace dataset or from local CSV.  
This step creates a local HF dataset for speed.


In [12]:

from datasets import Dataset, Audio
import pandas as pd

df = pd.read_csv(OUT_CSV)
ds = Dataset.from_pandas(df)
ds = ds.cast_column('file_name', Audio(sampling_rate=16000))
ds.save_to_disk(str(WORK_DIR / 'hf_dataset'))
print('Saved HuggingFace dataset at:', WORK_DIR / 'hf_dataset')


Saving the dataset (0/1 shards):   0%|          | 0/521 [00:00<?, ? examples/s]

Saved HuggingFace dataset at: /content/drive/MyDrive/thai_tts/work/hf_dataset



## Prepare MMS-TTS (Thai) checkpoint for training

This uses `convert_original_discriminator_checkpoint.py` from the recipe to create a trainable VITS-style checkpoint.


In [13]:

%cd /content/finetune-hf-vits

!python convert_original_discriminator_checkpoint.py   --language_code {LANG_CODE}   --pytorch_dump_folder_path /content/mms-tha-train

# (Optional) list files
!ls -lah /content/mms-tha-train

%cd /content


/content/finetune-hf-vits
2025-09-19 05:05:30.107951: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758258330.128067    5044 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758258330.134411    5044 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758258330.150262    5044 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758258330.150286    5044 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758258330.150290    5044 computation_placer.cc:1


## Training config

Adjust batch size, learning rate, max steps/epochs to your GPU and dataset size.


In [15]:

import json, os
cfg = {
  # Model arguments
  "model_name_or_path": "/content/mms-tha-train",
  
  # Data arguments 
  "dataset_name": "/content/drive/MyDrive/thai_tts/work/hf_dataset",
  "audio_column_name": "file_name",
  "text_column_name": "text",
  "train_split_name": "train",
  "eval_split_name": "train",
  
  # Training arguments
  "output_dir": "/content/tts-tha-checkpoints",
  "do_train": True,
  "per_device_train_batch_size": 8,
  "gradient_accumulation_steps": 2,
  "learning_rate": 0.0001,
  "max_steps": 20000,
  "logging_steps": 50,
  "save_steps": 1000,
  "eval_steps": 1000,
  "warmup_steps": 500,
  "fp16": True
}
os.makedirs('/content/config', exist_ok=True)
with open('/content/config/train_thai.json', 'w') as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)
print(open('/content/config/train_thai.json').read())


{
  "model_name_or_path": "/content/mms-tha-train",
  "dataset_name": "/content/drive/MyDrive/thai_tts/work/hf_dataset",
  "audio_column_name": "file_name",
  "text_column_name": "text",
  "train_split_name": "train",
  "eval_split_name": "train",
  "output_dir": "/content/tts-tha-checkpoints",
  "do_train": true,
  "per_device_train_batch_size": 8,
  "gradient_accumulation_steps": 2,
  "learning_rate": 0.0001,
  "max_steps": 20000,
  "logging_steps": 50,
  "save_steps": 1000,
  "eval_steps": 1000,
  "warmup_steps": 500,
  "fp16": true
}



## Launch training

This will start fine-tuning the Thai MMS-TTS model on your dataset.


In [16]:

%cd /content/finetune-hf-vits
!accelerate launch run_vits_finetuning.py /content/config/train_thai.json
%cd /content


/content/finetune-hf-vits
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Traceback (most recent call last):
  File "/content/finetune-hf-vits/run_vits_finetuning.py", line 22, in <module>
    from monotonic_align import maximum_path
  File "/content/finetune-hf-vits/monotonic_align/__init__.py", line 4, in <module>
    from .monotonic_align.core import maximum_path_c
ModuleNotFoundError: No module named 'monotonic_align.monotonic_align'
Traceback (most recent call last):
  File "/usr/local/bin/accelerate", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/accelerate_cli.py", li


## Quick inference test

Generate a sample audio using the fine-tuned checkpoint.


In [ ]:

from transformers import pipeline
import soundfile as sf

ckpt_dir = "/content/tts-tha-checkpoints"
tts = pipeline("text-to-speech", model=ckpt_dir, device=0 if torch.cuda.is_available() else -1)
sample_text = "สวัสดีครับ ยินดีที่ได้รู้จัก นี่คือระบบสังเคราะห์เสียงภาษาไทยที่ฝึกด้วยข้อมูลของเรา"
out = tts(sample_text)
sf.write('/content/sample_thai_tts.wav', out["audio"], out["sampling_rate"], subtype='PCM_16')
print('Saved:', '/content/sample_thai_tts.wav')



## Save to Drive (and optionally push to Hugging Face Hub)


In [ ]:

import shutil
drive_ckpt = str(WORK_DIR / 'checkpoints_mms_thai')
shutil.copytree('/content/tts-tha-checkpoints', drive_ckpt, dirs_exist_ok=True)
print('Copied checkpoints to:', drive_ckpt)

# Optional: push to Hub (uncomment and set your repo)
# !pip -q install huggingface_hub
# from huggingface_hub import HfApi, create_repo
# repo_id = "yourname/mms-thai-finetuned"
# create_repo(repo_id, private=True, exist_ok=True)
# !huggingface-cli login
# !git lfs install
# !git init /content/tts-tha-checkpoints
# %cd /content/tts-tha-checkpoints
# !git remote add origin https://huggingface.co/yourname/mms-thai-finetuned
# !git add . && git commit -m "Add Thai TTS fine-tuned" && git push -u origin main
# %cd /content



## 🎯 Training Tips & Troubleshooting

### 📊 Dataset Recommendations
- **Recommended**: 1–3 hours clean audio for decent voice cloning; 5–10+ hours for robust style control
- **Audio Quality**: Keep clips 1–12 seconds; remove background noise as much as possible
- **Consistency**: Use single speaker with consistent pronunciation and style
- **Format**: Audio will be automatically converted to 16kHz mono WAV

### ⚡ Performance Optimization
- **OOM Issues**: Reduce `per_device_train_batch_size` from 8 to 4 or 2, increase `gradient_accumulation_steps`
- **Slow Training**: Reduce `max_steps` initially (e.g., 5000) to validate pipeline, then scale up
- **GPU Usage**: Ensure T4/V100/A100 GPU is enabled in Colab runtime settings

### 🔧 Alternative Dataset Configuration
**Using CSV directly (instead of HF dataset):**
1. Set `cfg["dataset_name"] = "/path/to/metadata_clean.csv"`
2. Ensure CSV has columns: `file_name,text` (not `path,text`)
3. Re-run the training cell

### 🚨 Common Errors & Solutions
- **ModuleNotFoundError: monotonic_align** → ✅ Fixed by compilation step
- **Column 'fila_name' not found** → ✅ Fixed typo in dataset creation
- **Audio loading errors** → Check file paths and formats
- **CUDA out of memory** → Reduce batch size or use shorter clips

### 🎵 Using Your Trained Model
After training, use your model like this:
```python
from transformers import pipeline
import soundfile as sf

# Load fine-tuned model
tts = pipeline("text-to-speech", model="/content/tts-tha-checkpoints")

# Generate speech
text = "สวัสดีครับ นี่คือเสียงสังเคราะห์ภาษาไทยที่ฝึกมาใหม่"
output = tts(text)
sf.write("my_thai_voice.wav", output["audio"], output["sampling_rate"])
```

## 🏆 Success Checklist
- ✅ GPU enabled and detected
- ✅ Google Drive mounted successfully
- ✅ Dataset loaded and preprocessed
- ✅ HuggingFace dataset created
- ✅ MMS checkpoint prepared
- ✅ Training completed without errors
- ✅ Model generates clear Thai speech
- ✅ Checkpoints saved to Drive

**🎉 Happy Training! If this notebook helped you, please star the repository at:**  
**https://github.com/samiabat/thai-colab** ⭐
